# Train the install-tier model (Colab)

1. In `MyDrive/ML-Datasets/` (next to `googleplaystore_classification.csv`) put the project: either the **`install-tier-service` folder** or **`install-tier-service.zip`**. Delete any older copy first.
2. Runtime → Run all (about 15–20 minutes). A cell that fails stops the run with a message saying why.
3. Download `MyDrive/ML-Datasets/model_outputs.zip` and unzip it inside the project folder on your PC.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, shutil, glob
DATA_DIR = '/content/drive/MyDrive/ML-Datasets'
CSV = f'{DATA_DIR}/googleplaystore_classification.csv'
PROJECT = '/content/install-tier-service'

shutil.rmtree(PROJECT, ignore_errors=True)
if os.path.isdir(f'{DATA_DIR}/install-tier-service'):          # uploaded as a folder
    shutil.copytree(f'{DATA_DIR}/install-tier-service', PROJECT)
elif os.path.exists(f'{DATA_DIR}/install-tier-service.zip'):   # uploaded as the zip
    shutil.unpack_archive(f'{DATA_DIR}/install-tier-service.zip', '/content')
assert os.path.exists(f'{PROJECT}/train.py'), 'Project not found: put the install-tier-service folder or zip in MyDrive/ML-Datasets'
assert os.path.exists(CSV), f'Dataset not found: {CSV}'
%cd {PROJECT}
!pip install -q -r requirements.txt
!python -c "import common; print('Code version:', common.CODE_VERSION)"

## Raw Kaggle CSV (for the developer frequency table)

The processed CSV has no developer names, so the developer → frequency table is built from the raw Kaggle CSV. This cell uses `ML-Datasets/Google-Playstore.csv` if you uploaded it, otherwise downloads it from Kaggle. Without it training still runs, but every developer typed into the app counts as unseen.

In [ ]:
RAW = f'{DATA_DIR}/Google-Playstore.csv' if os.path.exists(f'{DATA_DIR}/Google-Playstore.csv') else None
if RAW is None:
    try:
        !pip install -q kagglehub
        import kagglehub
        RAW = glob.glob(kagglehub.dataset_download('gauthamp10/google-playstore-apps') + '/**/*.csv', recursive=True)[0]
    except Exception as e:
        print('Could not get the raw CSV:', e)
print('Raw CSV:', RAW)

## Train (automatic model-size search, ~20-30 minutes)

The report setting (100 trees, `min_samples_leaf` 5) scores macro-F1 0.4929 but is ~5 GB in memory, too big for a 6 GB laptop. This step tries several smaller forests (`trees x min_samples_leaf`), scores each on a **validation split taken from the training data** (the test split stays untouched), and keeps the one with the **highest macro-F1 whose model file fits `MAX_MODEL_GB`**. That one is refitted on the full training split and scored once on the test split.

Expect: `PASSED` (parity), a table with one line per candidate (macro-F1, size, recall per tier), `Chosen: ...`, then the test macro-F1, per-tier recall and precision, and `Saved ...`. Copy the table into the report.

- `MAX_MODEL_GB = 0.4` keeps the server plus retraining at about 2.3 GB extra RAM (laptop with ~6 GB).
- `CANDIDATES` can be changed, e.g. add `20x10` if you have more RAM (then raise `MAX_MODEL_GB`).

In [ ]:
CANDIDATES = '15x10,20x15,30x20,40x30'   # trees x min_samples_leaf
MAX_MODEL_GB = 0.4                         # laptop with ~6 GB RAM
RAW_ARG = f'--raw "{RAW}"' if RAW else ''
shutil.rmtree('registry', ignore_errors=True); shutil.rmtree('data', ignore_errors=True)
!python train.py --data "{CSV}" {RAW_ARG} --search {CANDIDATES} --max-model-gb {MAX_MODEL_GB}
assert os.path.exists('registry/current.json') and os.path.exists('data/test_X.npy'), 'Training failed: see the error above'

## Save the model to Drive

In [ ]:
!zip -qr "{DATA_DIR}/model_outputs.zip" registry data
!ls -lh "{DATA_DIR}/model_outputs.zip"